# Day 14 — PySpark Big Data and Data Quality (Databricks Free Edition) # 
Complete internship notebook — CSV ingestion, Spark transformations, joins, windows, UDF, reusable quality rules, quarantine, reports, assertions, large-scale simulation, and Delta Lake.



#Setup imports and configure volume #
Purpose: import Spark SQL APIs and configure one reusable directory. Update catalog/schema if your workspace uses different names.

In [0]:
# Spark is already created by Databricks. No pip install or manual SparkSession required.
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone
import json

# CHANGE these if Catalog Explorer shows different catalog/schema names.
CATALOG = "data"
SCHEMA = "default"
VOLUME = "day_14"
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
ORDERS_PATH = f"{BASE}/orders.csv"
CUSTOMERS_PATH = f"{BASE}/customers.csv"
OUTPUT_BASE = f"{BASE}/outputs"

print("Spark version:", spark.version)
print("Volume:", BASE)


Spark version: 4.2.0
Volume: /Volumes/data/default/day_14


# Verify the input CSV files were uploaded #
Check the actual files in the Unity Catalog volume before attempting to read them.

In [0]:
# dbutils.fs.ls works with fully qualified Unity Catalog volume paths.
try:
    uploaded = [f.name.rstrip("/") for f in dbutils.fs.ls(BASE)]
except Exception as exc:
    raise RuntimeError(f"Cannot access {BASE}. Check catalog/schema/volume name and permissions.") from exc

print("Files in volume:", uploaded)
missing_files = {"orders.csv", "customers.csv"} - set(uploaded)
assert not missing_files, f"Please upload these files first: {sorted(missing_files)}"
print("PASS: Both required input CSV files are available.")


Files in volume: ['customers.csv', 'orders.csv']
PASS: Both required input CSV files are available.


# Read CSV with explicit raw-text schema #
Keep values as strings initially: malformed dates, quantities or amounts should survive ingestion and be flagged in quality checks.

In [0]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("category", StringType(), True),
    StructField("status", StringType(), True)
])

orders = (spark.read.option("header", True)
          .option("mode", "PERMISSIVE")
          .schema(order_schema).csv(ORDERS_PATH))
customers = spark.read.option("header", True).csv(CUSTOMERS_PATH)

print("Order rows:", orders.count())
print("Customer rows:", customers.count())
orders.printSchema()
display(orders.limit(12))
display(customers.limit(12))


Order rows: 176
Customer rows: 110
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- category: string (nullable = true)
 |-- status: string (nullable = true)



order_id,customer_id,order_date,amount,quantity,category,status
O0001,C003,2026-10-09,2375.33,1,Grocery,delivered
O0002,C003,2026-10-01,1469.75,1,Electronics,paid
O0003,C006,2026-10-16,3327.63,4,Home,paid
O0004,C004,2026-10-11,2207.54,3,Grocery,shipped
O0005,C002,2026-10-21,1836.58,4,Home,paid
O0006,C004,2026-10-19,887.12,2,Electronics,paid
O0007,C004,2026-10-09,2442.36,3,Grocery,delivered
O0008,C005,2026-10-06,2202.65,1,Grocery,delivered
O0009,C004,2026-10-28,2952.02,5,Fashion,paid
O0010,C001,2026-10-26,2274.76,3,Grocery,paid


customer_id,customer_name,city
C001,Customer 1,Bengaluru
C002,Customer 2,Bengaluru
C003,Customer 3,Chennai
C004,Customer 4,Coimbatore
C005,Customer 5,Coimbatore
C006,Customer 6,Coimbatore
C007,Customer 7,Coimbatore
C008,Customer 8,Chennai
C009,Customer 9,Coimbatore
C010,Customer 10,Coimbatore


# Spark fundamentals: transformations versus actions #
select and filter define a plan; show, count, display, and writes trigger execution. Inspect the optimized execution plan with explain.

In [0]:
selected = orders.select("order_id", "customer_id", "amount", "category")
high_value = orders.filter(F.expr("try_cast(amount AS DOUBLE) >= 1000"))
print("Selected rows:")
display(selected.limit(10))
print("High-value raw orders:")
display(high_value.limit(10))
print("Physical and logical execution plan:")
high_value.explain(True)


Selected rows:


order_id,customer_id,amount,category
O0001,C003,2375.33,Grocery
O0002,C003,1469.75,Electronics
O0003,C006,3327.63,Home
O0004,C004,2207.54,Grocery
O0005,C002,1836.58,Home
O0006,C004,887.12,Electronics
O0007,C004,2442.36,Grocery
O0008,C005,2202.65,Grocery
O0009,C004,2952.02,Fashion
O0010,C001,2274.76,Grocery


High-value raw orders:


order_id,customer_id,order_date,amount,quantity,category,status
O0001,C003,2026-10-09,2375.33,1,Grocery,delivered
O0002,C003,2026-10-01,1469.75,1,Electronics,paid
O0003,C006,2026-10-16,3327.63,4,Home,paid
O0004,C004,2026-10-11,2207.54,3,Grocery,shipped
O0005,C002,2026-10-21,1836.58,4,Home,paid
O0007,C004,2026-10-09,2442.36,3,Grocery,delivered
O0008,C005,2026-10-06,2202.65,1,Grocery,delivered
O0009,C004,2026-10-28,2952.02,5,Fashion,paid
O0010,C001,2026-10-26,2274.76,3,Grocery,paid
O0011,C006,2026-10-20,2538.46,2,Grocery,delivered


Physical and logical execution plan:
== Parsed Logical Plan ==
'Filter (try_cast('amount as double) >= 1000)
+- Relation [order_id#10202,customer_id#10203,order_date#10204,amount#10205,quantity#10206,category#10207,status#10208] csv

== Analyzed Logical Plan ==
order_id: string, customer_id: string, order_date: string, amount: string, quantity: string, category: string, status: string
Filter (try_cast(amount#10205 as double) >= cast(1000 as double))
+- Relation [order_id#10202,customer_id#10203,order_date#10204,amount#10205,quantity#10206,category#10207,status#10208] csv

== Optimized Logical Plan ==
Filter (isnotnull(amount#10205) AND (try_cast(amount#10205 as double) >= 1000.0))
+- Relation [order_id#10202,customer_id#10203,order_date#10204,amount#10205,quantity#10206,category#10207,status#10208] csv

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonFilter (isnotnull(amount#10205) AND (try_cast(amount#10205 as double) >= 1000.0))
      +- PhotonRowToColumnar
 

# Aggregations and joins (learning examples) #
These examples use RAW orders; their totals are not yet trusted. Final business aggregates will be computed from clean records.

In [0]:
raw_summary = (orders.withColumn("amount_numeric", F.expr("try_cast(amount AS DOUBLE)"))
    .groupBy("category")
    .agg(F.count("*").alias("order_count"),
         F.round(F.sum("amount_numeric"), 2).alias("raw_amount_sum")))
display(raw_summary.orderBy("category"))

# LEFT JOIN retains an order even if no matching customer exists.
joined = orders.join(customers, on="customer_id", how="left")
display(joined.select("order_id", "customer_id", "customer_name", "city", "amount").limit(15))



category,order_count,raw_amount_sum
Electronics,44,69972.0
Fashion,38,66799.01
Grocery,44,102964.38
Home,50,86069.87


order_id,customer_id,customer_name,city,amount
O0001,C003,Customer 3,Chennai,2375.33
O0002,C003,Customer 3,Chennai,1469.75
O0003,C006,Customer 6,Coimbatore,3327.63
O0004,C004,Customer 4,Coimbatore,2207.54
O0005,C002,Customer 2,Bengaluru,1836.58
O0006,C004,Customer 4,Coimbatore,887.12
O0007,C004,Customer 4,Coimbatore,2442.36
O0008,C005,Customer 5,Coimbatore,2202.65
O0009,C004,Customer 4,Coimbatore,2952.02
O0010,C001,Customer 1,Bengaluru,2274.76


# Window functions and a Python UDF # 
Rank orders within each customer. Demonstrate a Python UDF, then the recommended native Spark expression.

In [0]:
from pyspark.sql.functions import udf

numeric_orders = orders.withColumn("amount_numeric", F.expr("try_cast(amount AS DOUBLE)"))
w = Window.partitionBy("customer_id").orderBy(
    F.col("amount_numeric").desc_nulls_last(), F.col("order_id"))
ranked = numeric_orders.withColumn("customer_amount_rank", F.row_number().over(w))
display(ranked.select("customer_id", "order_id", "amount_numeric", "customer_amount_rank").limit(20))

@udf(returnType="string")
def amount_band(amount):
    # A Python UDF can involve serialization between Spark and Python workers.
    if amount is None:
        return "missing"
    return "high" if amount >= 1000 else "low"

with_udf = numeric_orders.withColumn("band_udf", amount_band(F.col("amount_numeric")))
with_native = with_udf.withColumn(
    "band_native", F.when(F.col("amount_numeric").isNull(), "missing")
                       .when(F.col("amount_numeric") >= 1000, "high")
                       .otherwise("low"))
display(with_native.select("order_id", "amount_numeric", "band_udf", "band_native").limit(12))


customer_id,order_id,amount_numeric,customer_amount_rank
null,O0055,70.0,1
C001,O0030,2636.5,1
C001,O0032,2309.9,2
C001,O0010,2274.76,3
C001,O0056,2242.75,4
C001,O0026,1337.66,5
C001,O0029,1007.73,6
C001,O0048,202.93,7
C001,O0051,-12.0,8
C002,O0036,2879.01,1


order_id,amount_numeric,band_udf,band_native
O0001,2375.33,high,high
O0002,1469.75,high,high
O0003,3327.63,high,high
O0004,2207.54,high,high
O0005,1836.58,high,high
O0006,887.12,low,low
O0007,2442.36,high,high
O0008,2202.65,high,high
O0009,2952.02,high,high
O0010,2274.76,high,high


# Reusable quality-framework functions # 
Create reason codes for each invalid field, detect duplicate IDs, and keep original raw text. This is our custom validation framework (no external package needed).

In [0]:
REQUIRED_COLUMNS = {"order_id", "customer_id", "order_date", "amount", "quantity", "category", "status"}
VALID_STATUSES = ["pending", "paid", "shipped", "delivered", "cancelled"]

def validate_orders(raw_df):
    """Return rows with parsed types, error codes and a Boolean is_valid marker."""
    missing = REQUIRED_COLUMNS - set(raw_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Safe conversions: unparseable strings turn into nulls, not data loss.
    typed = (raw_df
        .withColumn("amount_numeric", F.expr("try_cast(amount AS DOUBLE)"))
        .withColumn("quantity_numeric", F.expr("try_cast(quantity AS INT)"))
        .withColumn("parsed_date", F.expr("try_cast(order_date AS DATE)")))

    # Mark EVERY row sharing a duplicate order ID; do not arbitrarily keep one.
    duplicated_ids = (typed.groupBy("order_id").count()
         .filter(F.col("count") > 1).select("order_id")
         .withColumn("duplicate_id", F.lit(True)))
    checked = typed.join(duplicated_ids, on="order_id", how="left")
    checked = checked.withColumn("duplicate_id", F.coalesce("duplicate_id", F.lit(False)))

    # Each condition is TRUE when the row fails that quality rule.
    rule_map = {
        "MISSING_ORDER_ID": F.col("order_id").isNull() | (F.trim("order_id") == ""),
        "MISSING_CUSTOMER_ID": F.col("customer_id").isNull() | (F.trim("customer_id") == ""),
        "INVALID_DATE": F.col("parsed_date").isNull(),
        "INVALID_AMOUNT": F.col("amount_numeric").isNull() | (F.col("amount_numeric") < 0) | F.isnan("amount_numeric"),
        "INVALID_QUANTITY": F.col("quantity_numeric").isNull() | (F.col("quantity_numeric") <= 0),
        "MISSING_CATEGORY": F.col("category").isNull() | (F.trim("category") == ""),
        "INVALID_STATUS": F.col("status").isNull() | (~F.col("status").isin(VALID_STATUSES)),
        "DUPLICATE_ORDER_ID": F.col("duplicate_id")
    }
    # array_compact removes null entries from the array of triggered errors.
    error_items = [F.when(condition, F.lit(reason)) for reason, condition in rule_map.items()]
    flagged = (checked
        .withColumn("quality_errors", F.array_compact(F.array(*error_items)))
        .withColumn("is_valid", F.size("quality_errors") == 0)
        .drop("duplicate_id"))
    return flagged


def split_valid_rejected(flagged_df):
    """Produce typed accepted rows and preserved, annotated rejected rows."""
    clean = (flagged_df.filter(F.col("is_valid"))
        .select("order_id", "customer_id", F.col("parsed_date").alias("order_date"),
                F.col("amount_numeric").alias("amount"),
                F.col("quantity_numeric").alias("quantity"), "category", "status"))
    quarantine = flagged_df.filter(~F.col("is_valid"))
    return clean, quarantine


# Execute validation and examine rejected rows # 
The source sample intentionally includes invalid amounts, a malformed date, zero quantity, invalid status, missing customer, and duplicate order IDs.

In [0]:
flagged_orders = validate_orders(orders).cache()
valid_orders, rejected_orders = split_valid_rejected(flagged_orders)

total_count = flagged_orders.count()  # Materialize cache
valid_count = valid_orders.count()
rejected_count = rejected_orders.count()
print(f"Total={total_count}, Valid={valid_count}, Rejected={rejected_count}")

# Show diagnostic reasons and original source text.
display(rejected_orders.select("order_id", "customer_id", "order_date", "amount",
                               "quantity", "status", "quality_errors"))


Total=176, Valid=169, Rejected=7


order_id,customer_id,order_date,amount,quantity,status,quality_errors
O0001,C003,2026-10-09,2375.33,1,delivered,List(DUPLICATE_ORDER_ID)
O0051,C001,2026-10-03,-12,1,paid,List(INVALID_AMOUNT)
O0052,C004,bad-date,500,2,paid,List(INVALID_DATE)
O0053,C002,2026-10-05,120,0,paid,List(INVALID_QUANTITY)
O0054,C002,2026-10-05,110,1,unknown,List(INVALID_STATUS)
O0001,C006,2026-10-07,90,1,paid,List(DUPLICATE_ORDER_ID)
O0055,null,2026-10-07,70,1,paid,List(MISSING_CUSTOMER_ID)


# Quality metrics and report # 
Count all rule failures efficiently by exploding each error array; one invalid row can have multiple errors. Record a UTC timestamp and reconciliation metrics.

In [0]:
rule_failure_df = (flagged_orders
    .select(F.explode_outer("quality_errors").alias("rule_name"))
    .filter(F.col("rule_name").isNotNull())
    .groupBy("rule_name").agg(F.count("*").alias("failed_rows"))
    .orderBy(F.desc("failed_rows"), "rule_name"))
display(rule_failure_df)

quality_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "input": ORDERS_PATH,
    "total_rows": total_count,
    "valid_rows": valid_count,
    "rejected_rows": rejected_count,
    "valid_percent": round(100.0 * valid_count / total_count, 2) if total_count else 0.0,
    "quality_gate_passed": rejected_count == 0,
    "reconciliation_passed": total_count == valid_count + rejected_count,
    "rule_failures": {r["rule_name"]: r["failed_rows"] for r in rule_failure_df.collect()}
}
print(json.dumps(quality_report, indent=2))


rule_name,failed_rows
DUPLICATE_ORDER_ID,2
INVALID_AMOUNT,1
INVALID_DATE,1
INVALID_QUANTITY,1
INVALID_STATUS,1
MISSING_CUSTOMER_ID,1


{
  "generated_at_utc": "2026-10-28T05:26:35.005233+00:00",
  "input": "/Volumes/data/default/day_14/orders.csv",
  "total_rows": 176,
  "valid_rows": 169,
  "rejected_rows": 7,
  "valid_percent": 96.02,
  "quality_gate_passed": false,
  "reconciliation_passed": true,
  "rule_failures": {
    "DUPLICATE_ORDER_ID": 2,
    "INVALID_AMOUNT": 1,
    "INVALID_DATE": 1,
    "INVALID_QUANTITY": 1,
    "INVALID_STATUS": 1,
    "MISSING_CUSTOMER_ID": 1
  }
}


# Persist valid, rejected and aggregate outputs # 
Write clean Parquet, quarantine JSON, category CSV and a JSON report into separate volume folders. overwrite replaces prior demonstration outputs, so do not use for production history.

In [0]:
valid_orders.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/valid_orders")
rejected_orders.write.mode("overwrite").json(f"{OUTPUT_BASE}/rejected_orders")

category_summary = (valid_orders.groupBy("category")
    .agg(F.count("*").alias("order_count"),
         F.round(F.sum("amount"), 2).alias("total_order_amount")))
category_summary.write.mode("overwrite").option("header", True).csv(f"{OUTPUT_BASE}/category_summary")

dbutils.fs.put(f"{OUTPUT_BASE}/quality_report.json", json.dumps(quality_report, indent=2), True)
print("Output root:", OUTPUT_BASE)
display(category_summary.orderBy(F.desc("total_order_amount")))


Wrote 453 bytes.
Output root: /Volumes/data/default/day_14/outputs


category,order_count,total_order_amount
Grocery,42,100469.05
Home,46,85811.87
Electronics,44,69972.0
Fashion,37,66299.01


# Automated post-write assertions #
Validate actual persisted output, not only intermediate in-memory variables. The original raw source may fail the strict quality gate while the accepted dataset passes every assertion.

In [0]:
persisted_valid = spark.read.parquet(f"{OUTPUT_BASE}/valid_orders")
persisted_rejected = spark.read.json(f"{OUTPUT_BASE}/rejected_orders")

assert total_count == valid_count + rejected_count, "Input/output row count mismatch"
assert persisted_valid.count() == valid_count, "Persisted clean row count mismatch"
assert persisted_rejected.count() == rejected_count, "Persisted quarantine row count mismatch"
assert persisted_valid.filter(F.col("order_id").isNull() | (F.trim("order_id") == "")).count() == 0
assert persisted_valid.filter(F.col("customer_id").isNull() | (F.trim("customer_id") == "")).count() == 0
assert persisted_valid.filter(F.col("order_date").isNull()).count() == 0
assert persisted_valid.filter(F.col("amount").isNull() | (F.col("amount") < 0) | F.isnan("amount")).count() == 0
assert persisted_valid.filter(F.col("quantity").isNull() | (F.col("quantity") <= 0)).count() == 0
assert persisted_valid.filter(~F.col("status").isin(VALID_STATUSES) | F.col("status").isNull()).count() == 0
assert persisted_valid.groupBy("order_id").count().filter("count > 1").count() == 0

print("PASS: row reconciliation, persisted output counts, required fields, types, amounts, statuses and uniqueness.")


PASS: row reconciliation, persisted output counts, required fields, types, amounts, statuses and uniqueness.


# Unit-style tests for reusable rules # 
Create a tiny synthetic DataFrame with duplicate ID, negative amount and a clean row. Check expected outcomes. These tests do not depend on the input CSV.

In [0]:
test_rows = [
    ("A", "C1", "2026-10-01", "100", "2", "Home", "paid"),
    ("A", "C2", "2026-10-01", "150", "1", "Home", "paid"),
    ("B", "C3", "2026-10-02", "-1", "1", "Home", "paid"),
    ("C", "C4", "2026-10-03", "70", "1", "Home", "paid")
]
test_df = spark.createDataFrame(test_rows, order_schema)
flagged_test = validate_orders(test_df)
assert flagged_test.filter("is_valid").count() == 1
assert flagged_test.filter("NOT is_valid").count() == 3
assert flagged_test.filter(F.array_contains("quality_errors", "DUPLICATE_ORDER_ID")).count() == 2

try:
    validate_orders(spark.createDataFrame([("A",)], ["order_id"]))
    raise AssertionError("Expected missing-column ValueError")
except ValueError:
    pass
print("PASS: duplicate, negative amount and required-schema tests.")


PASS: duplicate, negative amount and required-schema tests.


# Generate 100,000 synthetic rows using Spark # 
No local Python CSV generator needed. Spark creates the dataset in the cloud workspace; this generates valid example records. To control Free Edition quotas, change N to 10,000 for a smaller run.

In [0]:
N = 100000  # Set to 10000 if your Free Edition quotas are tight.
synthetic = (spark.range(1, N + 1)
    .select(
        F.format_string("O%08d", F.col("id")).alias("order_id"),
        F.format_string("C%04d", (F.col("id") % 500) + 1).alias("customer_id"),
        F.date_format(F.date_add(F.lit("2026-10-01").cast("date"), (F.col("id") % 28).cast("int")), "yyyy-MM-dd").alias("order_date"),
        F.format_string("%.2f", F.lit(20.0) + (F.col("id") % 3980).cast("double")).alias("amount"),
        ((F.col("id") % 7) + 1).cast("string").alias("quantity"),
        F.when(F.col("id") % 4 == 0, "Electronics")
         .when(F.col("id") % 4 == 1, "Fashion")
         .when(F.col("id") % 4 == 2, "Grocery")
         .otherwise("Home").alias("category"),
        F.when(F.col("id") % 3 == 0, "paid")
         .when(F.col("id") % 3 == 1, "shipped")
         .otherwise("delivered").alias("status")
    ))
large_path = f"{BASE}/synthetic_large_orders"
synthetic.write.mode("overwrite").option("header", True).csv(large_path)
print("Created CSV dataset:", large_path)
print("Rows generated:", synthetic.count())


Created CSV dataset: /Volumes/data/default/day_14/synthetic_large_orders
Rows generated: 100000


# Process the larger dataset through the same framework # 
Demonstrates modular code reuse. Keep outputs separate from the 56-row sample so you can submit both results.

In [0]:
large_raw = (spark.read.option("header", True).schema(order_schema).csv(large_path))
large_flagged = validate_orders(large_raw)
large_valid, large_rejected = split_valid_rejected(large_flagged)
large_total = large_raw.count()
large_good = large_valid.count()
large_bad = large_rejected.count()
assert large_total == large_good + large_bad
large_valid.write.mode("overwrite").parquet(f"{OUTPUT_BASE}/large_valid_orders")
print(f"Large dataset: total={large_total:,}, valid={large_good:,}, rejected={large_bad:,}")


Large dataset: total=100,000, valid=100,000, rejected=0


# Save clean records as a Delta table # 
Delta adds transactional table management to your analytical workflow. The table is created in your selected catalog and schema.

In [0]:
TABLE_NAME = f"{CATALOG}.{SCHEMA}.day14_clean_orders"
(valid_orders.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME))
print("Saved Delta table:", TABLE_NAME)
display(spark.sql(f"SELECT category, COUNT(*) AS orders, ROUND(SUM(amount), 2) AS order_amount FROM {TABLE_NAME} GROUP BY category ORDER BY order_amount DESC"))


Saved Delta table: data.default.day14_clean_orders


category,orders,order_amount
Grocery,42,100469.05
Home,46,85811.87
Electronics,44,69972.0
Fashion,37,66299.01


# Inspect outputs and finish #
Each Spark write creates a directory with one or more part-* files. The report is a single JSON file. Unpersist the cached source DataFrame at the end; do not stop the managed Databricks Spark session.

In [0]:
print("Files/folders at output root:")
for item in dbutils.fs.ls(OUTPUT_BASE):
    print(item.path)
print("Quality report:\n", dbutils.fs.head(f"{OUTPUT_BASE}/quality_report.json", 10000))
flagged_orders.unpersist()
print("DONE: Day 14 notebook completed.")


Files/folders at output root:
dbfs:/Volumes/data/default/day_14/outputs/category_summary/
dbfs:/Volumes/data/default/day_14/outputs/large_valid_orders/
dbfs:/Volumes/data/default/day_14/outputs/quality_report.json
dbfs:/Volumes/data/default/day_14/outputs/rejected_orders/
dbfs:/Volumes/data/default/day_14/outputs/valid_orders/
Quality report:
 {
  "generated_at_utc": "2026-10-28T05:26:35.005233+00:00",
  "input": "/Volumes/data/default/day_14/orders.csv",
  "total_rows": 176,
  "valid_rows": 169,
  "rejected_rows": 7,
  "valid_percent": 96.02,
  "quality_gate_passed": false,
  "reconciliation_passed": true,
  "rule_failures": {
    "DUPLICATE_ORDER_ID": 2,
    "INVALID_AMOUNT": 1,
    "INVALID_DATE": 1,
    "INVALID_QUANTITY": 1,
    "INVALID_STATUS": 1,
    "MISSING_CUSTOMER_ID": 1
  }
}
DONE: Day 14 notebook completed.


In [0]:
BASE = "/Volumes/data/default/day_14"
OUTPUT_BASE = f"{BASE}/outputs"

# Load the large dataset
large_df = spark.read.parquet(
    f"{OUTPUT_BASE}/large_valid_orders"
)

# Check the number of records and columns
print("Total rows:", large_df.count())
print("Columns:", large_df.columns)

# Display the actual records
display(large_df.limit(20))

Total rows: 100000
Columns: ['order_id', 'customer_id', 'order_date', 'amount', 'quantity', 'category', 'status']


order_id,customer_id,order_date,amount,quantity,category,status
O00050001,C0002,2026-10-22,2261.0,1,Fashion,paid
O00050002,C0003,2026-10-23,2262.0,2,Grocery,shipped
O00050003,C0004,2026-10-24,2263.0,3,Home,delivered
O00050004,C0005,2026-10-25,2264.0,4,Electronics,paid
O00050005,C0006,2026-10-26,2265.0,5,Fashion,shipped
O00050006,C0007,2026-10-27,2266.0,6,Grocery,delivered
O00050007,C0008,2026-10-28,2267.0,7,Home,paid
O00050008,C0009,2026-10-01,2268.0,1,Electronics,shipped
O00050009,C0010,2026-10-02,2269.0,2,Fashion,delivered
O00050010,C0011,2026-10-03,2270.0,3,Grocery,paid


In [0]:
csv_output = f"{OUTPUT_BASE}/large_valid_orders_csv"

# Convert Parquet data to CSV
large_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(csv_output)

print("CSV export location:", csv_output)

for item in dbutils.fs.ls(csv_output):
    print(item.name, item.size)

CSV export location: /Volumes/data/default/day_14/outputs/large_valid_orders_csv
_SUCCESS 0
_committed_7648458454371651187 113
_started_7648458454371651187 0
part-00000-tid-7648458454371651187-118ba8b6-f7c5-4166-8713-f29ebe7232b6-333-1-c000.csv 5164652
